# SpaNCy-Shift Explorer

Adaptive per-sample shift correction for CyCIF batch normalization.
Combines CycleDegradationModel (affine) with AdaptiveShiftModel (bimodal-aware shifts) and 20D MMD for kBET.

**Sections:**
0. Colab Setup
1. Load & Inspect Data
2. Bimodal Marker Detection
3. Single Model Training
4. Inference & Output Inspection
5. Batch adj-R² Diagnostics
6. Positive Population Preservation Check
7. Histogram Comparison
8. kBET Evaluation

## 0. Colab Setup

Run first on Google Colab. Installs dependencies and uploads `spancy_shift.py`.

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q anndata scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift.py
from google.colab import files
uploaded = files.upload()

import os
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found — upload it first'
print(f"spancy_shift.py uploaded ({os.path.getsize('spancy_shift.py'):,} bytes)")

In [ ]:
import importlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt

import spancy_shift
from spancy_shift import (
    load_adata, get_spatial_coords, get_scene_ids,
    assign_marker_cycles, log1p_scale, DEFAULT_CYCLE_CONFIG,
    detect_bimodal_markers, SpaNCyShift, train,
    normalize_adata, sample_mode_align,
    positive_population_table, per_marker_batch_r2,
)

# After re-uploading spancy_shift.py, run:
# importlib.reload(spancy_shift); from spancy_shift import *

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Download PRAD dataset
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load & Inspect Data

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch'].unique().tolist())}")
print(f"\nobs columns: {list(adata.obs.columns)}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)

In [ ]:
# Per-batch mean intensity overview
batch_vals = adata.obs['batch'].values
unique_batches = sorted(adata.obs['batch'].unique())

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

axes[0].boxplot(
    [X_raw[:, i] for i in range(X_raw.shape[1])],
    labels=marker_names, vert=True
)
axes[0].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[0].set_title('Raw marker distributions (all cells)')
axes[0].set_ylabel('Intensity')

for b in unique_batches:
    mask = batch_vals == b
    axes[1].plot(X_raw[mask].mean(axis=0), marker='o', markersize=3, label=str(b), alpha=0.8)
axes[1].set_xticks(range(len(marker_names)))
axes[1].set_xticklabels(marker_names, rotation=90, fontsize=8)
axes[1].set_title('Per-batch mean intensity')
axes[1].legend(fontsize=6, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Bimodal Marker Detection

SpaNCy-Shift uses `_find_peaks()` to adaptively classify each marker as bimodal or unimodal
via **per-batch voting** — a marker is bimodal only if ≥50% of batches independently show ≥2
prominent histogram peaks. This prevents batch-separated unimodal distributions (e.g. ChromA,
where different batches sit at different positions) from being falsely classified as bimodal
from the pooled global histogram.

- **Bimodal**: ≥2 peaks in majority of batches → separate neg/pos peak shifts per sample
- **Unimodal**: 1 peak in most batches → single mean shift per sample

In [ ]:
# Scale data first (detect_bimodal_markers works on log1p-scaled space)
X_scaled, scaler_preview = log1p_scale(X_raw)
print(f'Scaled range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]')

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5  # marker must be bimodal in >= 50% of batches

# Get batch codes for per-batch voting
batch_codes_preview = adata.obs['batch'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_scaled, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.02,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"\nBimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")

print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")

In [ ]:
# Visualize global histograms with detected peaks and thresholds
from scipy.ndimage import gaussian_filter1d

n_cols = 5
n_rows = int(np.ceil(len(marker_names) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows))
axes = axes.flatten()

for m, mname in enumerate(marker_names):
    ax = axes[m]
    col = X_scaled[:, m]
    lo, hi = np.percentile(col, [1, 99])
    bins = np.linspace(lo, hi, 151)
    counts, edges = np.histogram(col, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    smoothed = gaussian_filter1d(counts.astype(float), sigma=2)

    ax.fill_between(centers, smoothed, alpha=0.4,
                    color='steelblue' if marker_is_bimodal[m] else 'salmon')
    ax.plot(centers, smoothed, 'k-', linewidth=1)

    if marker_is_bimodal[m]:
        ax.axvline(thresholds[m], color='red', linestyle='--', linewidth=1.5,
                   label=f'thresh={thresholds[m]:.2f}')
        ax.legend(fontsize=6)

    label = 'BIMODAL' if marker_is_bimodal[m] else 'unimodal'
    ax.set_title(f'{mname}\n({label})', fontsize=8,
                 fontweight='bold' if marker_is_bimodal[m] else 'normal')
    ax.set_xlabel('scaled value', fontsize=7)
    ax.tick_params(labelsize=6)

for ax in axes[len(marker_names):]:
    ax.set_visible(False)

plt.suptitle('Global Marker Distributions — Bimodal Detection', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Single Model Training

**Losses:**
- `L_identity` (Huber × 2.0): keep shifts close to affine baseline
- `L_align` (quantile × 1.0): align 10th/25th percentiles across samples
- `L_mmd` (20D RBF × 0.5, warmed up): align batch distributions for kBET

In [ ]:
N_EPOCHS = 10  # quick test; use 100 for production

model, trained_scaler, trained_marker_cycles, history = train(
    adata,
    cycle_config=DEFAULT_CYCLE_CONFIG,
    n_epochs=N_EPOCHS,
    lr=1e-3,
    cells_per_step=16000,
    device_str=DEVICE,
    warmup_epochs=3,
    mmd_ramp_start=3,
    mmd_ramp_end=7,
    w_mmd=0.5,
    w_recon=1.0,
    w_identity=0.5,
    w_align=0.5,
    mmd_samples=256,
    bimodal_min_batch_frac=0.5,
)

print(f'\nTraining complete. Final loss: {history["loss"][-1]:.4f}')
print(f'  recon={history["recon"][-1]:.4f}  identity={history["identity"][-1]:.4f}  '
      f'align={history["align"][-1]:.4f}  mmd={history["mmd"][-1]:.4f}')

In [ ]:
# Loss curves
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

axes[0, 0].plot(history['loss'], 'k-', linewidth=1.5)
axes[0, 0].set_title('Total Loss')

axes[0, 1].plot(history['recon'], color='darkorange', linewidth=1.5)
axes[0, 1].set_title('Recon Loss (anchors CycleDegradation)')

axes[0, 2].plot(history['identity'], 'b-', linewidth=1.5)
axes[0, 2].set_title('Identity Loss (anchors shifts)')

axes[1, 0].plot(history['mmd'], 'r-', linewidth=1.5)
ax_w = axes[1, 0].twinx()
ax_w.plot(history['mmd_weight'], 'g--', linewidth=1, alpha=0.6)
ax_w.set_ylabel('MMD weight', color='green', fontsize=9)
axes[1, 0].set_title('MMD Loss + Weight Ramp')

axes[1, 1].plot(history['align'], 'c-', linewidth=1.5)
axes[1, 1].set_title('Quantile Alignment Loss')

# Shift magnitudes
with torch.no_grad():
    s_neg = model.shift_model.shift_neg.weight.cpu().numpy()
im = axes[1, 2].imshow(np.abs(s_neg), aspect='auto', cmap='YlOrRd')
axes[1, 2].set_xlabel('Marker')
axes[1, 2].set_ylabel('Sample')
axes[1, 2].set_xticks(range(len(marker_names)))
axes[1, 2].set_xticklabels(marker_names, rotation=90, fontsize=7)
axes[1, 2].set_title('|shift_neg| per sample × marker')
plt.colorbar(im, ax=axes[1, 2], label='|shift|')

for ax in axes.flatten()[:5]:
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.suptitle('SpaNCy-Shift Training', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Inference & Output Inspection

Run inference with both affine-only and shift modes.
Use `shift_alpha` to blend between affine (0) and full shift (1).

In [ ]:
# Affine-only baseline
adata = normalize_adata(
    adata, model, trained_scaler, trained_marker_cycles,
    device_str=DEVICE, mode='affine',
    layer_name='normalized_affine',
)

# Shift at several alpha values
ALPHA_VALUES = [0.3, 1.0]
for alpha in ALPHA_VALUES:
    adata = normalize_adata(
        adata, model, trained_scaler, trained_marker_cycles,
        device_str=DEVICE, mode='shift', shift_alpha=alpha,
        layer_name=f'normalized_shift_a{alpha:.1f}',
    )

# Default 'normalized' = shift at alpha=1.0
adata.layers['normalized'] = adata.layers['normalized_shift_a1.0'].copy()

print(f'\nLayers: {list(adata.layers.keys())}')

In [ ]:
# Compare raw vs affine vs shift for bimodal markers
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]
plot_markers = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names + marker_names[:max(0, 4 - len(bimodal_names))]

layers_to_show = [
    (None, 'Raw'),
    ('normalized_affine', 'Affine only'),
    ('normalized_shift_a1.0', 'Shift (α=1.0)'),
]

fig, axes = plt.subplots(len(plot_markers), len(layers_to_show),
                          figsize=(6 * len(layers_to_show), 4 * len(plot_markers)))
if len(plot_markers) == 1:
    axes = axes.reshape(1, -1)

for row, mname in enumerate(plot_markers):
    m_idx = marker_names.index(mname)
    for col, (layer_key, title) in enumerate(layers_to_show):
        ax = axes[row, col]
        data = X_raw if layer_key is None else adata.layers[layer_key]
        for b in unique_batches:
            mask = batch_vals == b
            vals = np.log1p(np.clip(data[mask, m_idx], 0, None))
            ax.hist(vals, bins=80, alpha=0.4, density=True, label=str(b))
        ax.set_title(f'{mname} — {title}')
        ax.set_xlabel('log1p intensity')
        if col == 0:
            ax.set_ylabel('Density')
        if row == 0 and col == 0:
            ax.legend(fontsize=6, title='Batch', ncol=2)

plt.suptitle('Bimodal Marker Distributions: Raw → Affine → Shift', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Batch adj-R² Diagnostics

Adjusted R² for each marker regressed on batch labels.
Lower = batch effect better removed.

In [ ]:
batch_arr = adata.obs['batch'].values

r2_raw     = per_marker_batch_r2(X_raw,                              batch_arr, marker_names)
r2_affine  = per_marker_batch_r2(adata.layers['normalized_affine'],  batch_arr, marker_names)
r2_shift10 = per_marker_batch_r2(adata.layers['normalized'],         batch_arr, marker_names)

r2_compare = (
    r2_raw.rename(columns={'adj_r2': 'raw'})
    .merge(r2_affine.rename(columns={'adj_r2': 'affine'}), on='marker')
    .merge(r2_shift10.rename(columns={'adj_r2': 'shift_a1.0'}), on='marker')
)
cols = ['raw', 'affine', 'shift_a1.0']

if 'normalized_shift_a0.3' in adata.layers:
    r2_shift03 = per_marker_batch_r2(adata.layers['normalized_shift_a0.3'], batch_arr, marker_names)
    r2_compare = r2_compare.merge(r2_shift03.rename(columns={'adj_r2': 'shift_a0.3'}), on='marker')
    cols = ['raw', 'affine', 'shift_a0.3', 'shift_a1.0']

print('Per-Marker Batch adj-R² (lower = better):\n')
print(r2_compare.to_string(index=False, float_format='%.4f'))
print(f'\nMean adj-R²:')
for col in cols:
    print(f'  {col:>14s}: {r2_compare[col].mean():.4f}')

In [ ]:
COLOR_MAP = {
    'raw':        'salmon',
    'affine':     'steelblue',
    'shift_a0.3': 'mediumpurple',
    'shift_a1.0': 'forestgreen',
}
LABEL_MAP = {
    'raw':        'Raw',
    'affine':     'Affine only',
    'shift_a0.3': 'Shift (α=0.3)',
    'shift_a1.0': 'Shift (α=1.0)',
}

fig, ax = plt.subplots(figsize=(20, 5))
x = np.arange(len(r2_compare))
n = len(cols)
w = 0.7 / n
offsets = np.linspace(-(n - 1) / 2 * w, (n - 1) / 2 * w, n)

for offset, col in zip(offsets, cols):
    ax.bar(x + offset, r2_compare[col], w,
           label=LABEL_MAP[col], color=COLOR_MAP[col], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(r2_compare['marker'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Adjusted R²')
ax.set_title('Per-Marker Batch adj-R² — lower is better')
ax.axhline(0.05, color='red', linestyle='--', alpha=0.5, label='Target (0.05)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Positive Population Preservation Check

% positive cells per marker per sample using Otsu's threshold.
Delta = normalized − raw. Target: |delta| < 5% per marker.

- **6a**: Affine only
- **6b**: Shift α=0.3 (if available)
- **6c**: Shift α=1.0 (full)
- **6d**: Violin plot — all methods compared side by side

### 6a. Positive Population Preservation — Affine only

In [ ]:
pop_table_affine = positive_population_table(
    adata, norm_layer='normalized_affine', sample_col='sample_id'
)

summary_affine = pop_table_affine.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (affine vs raw):')
print('=' * 65)
print(summary_affine.to_string())
print(f'\nOverall mean |delta|: {pop_table_affine["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table_affine["delta"].abs().max():.2f}%')

In [ ]:
pivot_affine = pop_table_affine.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_affine.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot_affine.columns)))
ax.set_xticklabels(pivot_affine.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_affine.index)))
ax.set_yticklabels(pivot_affine.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: Affine vs Raw')
plt.tight_layout()
plt.show()

In [ ]:
plot_markers_pop = bimodal_names[:4] if len(bimodal_names) >= 4 else bimodal_names[:2]

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]

for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table_affine[pop_table_affine['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (Affine)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Positive Population Preservation — Affine only', fontsize=13)
plt.tight_layout()
plt.show()

### 6b. Positive Population Preservation — Shift α=0.3

In [ ]:
if 'normalized_shift_a0.3' in adata.layers:
    pop_table_shift03 = positive_population_table(
        adata, norm_layer='normalized_shift_a0.3', sample_col='sample_id'
    )
    summary_shift03 = pop_table_shift03.groupby('marker').agg(
        mean_pct_raw=('pct_pos_raw',   'mean'),
        mean_pct_norm=('pct_pos_norm',  'mean'),
        mean_abs_delta=('delta', lambda x: x.abs().mean()),
        max_abs_delta=('delta',  lambda x: x.abs().max()),
    ).round(2)
    print('Positive Population Preservation (shift_a0.3 vs raw):')
    print('=' * 65)
    print(summary_shift03.to_string())
    print(f'\nOverall mean |delta|: {pop_table_shift03["delta"].abs().mean():.2f}%')
    print(f'Overall max  |delta|: {pop_table_shift03["delta"].abs().max():.2f}%')

    pivot_shift03 = pop_table_shift03.pivot(index='marker', columns='sample', values='delta')
    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(pivot_shift03.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
    ax.set_xticks(range(len(pivot_shift03.columns)))
    ax.set_xticklabels(pivot_shift03.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot_shift03.index)))
    ax.set_yticklabels(pivot_shift03.index, fontsize=9)
    plt.colorbar(im, label='Δ % positive (norm − raw)')
    ax.set_title('Positive Population Change: Shift α=0.3 vs Raw')
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, len(plot_markers_pop),
                              figsize=(5 * len(plot_markers_pop), 5))
    if len(plot_markers_pop) == 1:
        axes = [axes]
    for ax, marker in zip(axes, plot_markers_pop):
        sub = pop_table_shift03[pop_table_shift03['marker'] == marker]
        ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
        lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
        ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
        ax.set_xlabel('% Positive (Raw)')
        ax.set_ylabel('% Positive (Shift α=0.3)')
        ax.set_title(marker)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.suptitle('Positive Population Preservation — Shift α=0.3', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('normalized_shift_a0.3 not available — skipping 6b.')
    pop_table_shift03 = None

### 6c. Positive Population Preservation — Shift α=1.0

In [ ]:
pop_table_shift10 = positive_population_table(
    adata, norm_layer='normalized', sample_col='sample_id'
)

summary_shift10 = pop_table_shift10.groupby('marker').agg(
    mean_pct_raw=('pct_pos_raw',   'mean'),
    mean_pct_norm=('pct_pos_norm',  'mean'),
    mean_abs_delta=('delta', lambda x: x.abs().mean()),
    max_abs_delta=('delta',  lambda x: x.abs().max()),
).round(2)

print('Positive Population Preservation (shift_a1.0 vs raw):')
print('=' * 65)
print(summary_shift10.to_string())
print(f'\nOverall mean |delta|: {pop_table_shift10["delta"].abs().mean():.2f}%')
print(f'Overall max  |delta|: {pop_table_shift10["delta"].abs().max():.2f}%')

pivot_shift10 = pop_table_shift10.pivot(index='marker', columns='sample', values='delta')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(pivot_shift10.values, cmap='RdBu_r', vmin=-10, vmax=10, aspect='auto')
ax.set_xticks(range(len(pivot_shift10.columns)))
ax.set_xticklabels(pivot_shift10.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot_shift10.index)))
ax.set_yticklabels(pivot_shift10.index, fontsize=9)
plt.colorbar(im, label='Δ % positive (norm − raw)')
ax.set_title('Positive Population Change: Shift α=1.0 vs Raw')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(plot_markers_pop),
                          figsize=(5 * len(plot_markers_pop), 5))
if len(plot_markers_pop) == 1:
    axes = [axes]
for ax, marker in zip(axes, plot_markers_pop):
    sub = pop_table_shift10[pop_table_shift10['marker'] == marker]
    ax.scatter(sub['pct_pos_raw'], sub['pct_pos_norm'], alpha=0.7, s=50)
    lims = [0, max(sub['pct_pos_raw'].max(), sub['pct_pos_norm'].max()) + 5]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='y=x')
    ax.set_xlabel('% Positive (Raw)')
    ax.set_ylabel('% Positive (Shift α=1.0)')
    ax.set_title(marker)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle('Positive Population Preservation — Shift α=1.0', fontsize=13)
plt.tight_layout()
plt.show()

### 6d. Violin Plot — Positive Population Preservation (All Methods)

One violin per marker per method, showing the distribution of % positive cells across 20 samples.
White dot = mean, black line = median. Blue marker titles = bimodal markers.

In [ ]:
from matplotlib.patches import Patch

METHOD_ORDER_SHIFT  = ['Raw', 'Affine', 'Shift α=0.3', 'Shift α=1.0']
METHOD_COLORS_SHIFT = {
    'Raw':        '#888888',
    'Affine':     '#4878d0',
    'Shift α=0.3': '#956cb4',
    'Shift α=1.0': '#6acc65',
}

# Build unified long-format table
frames = []
for mname, pt in [('Affine', pop_table_affine), ('Shift α=1.0', pop_table_shift10)]:
    tmp = pt[['marker', 'sample', 'pct_pos_norm']].copy()
    tmp['method'] = mname
    frames.append(tmp)

if pop_table_shift03 is not None:
    tmp = pop_table_shift03[['marker', 'sample', 'pct_pos_norm']].copy()
    tmp['method'] = 'Shift α=0.3'
    frames.append(tmp)

raw_tmp = pop_table_affine[['marker', 'sample', 'pct_pos_raw']].rename(
    columns={'pct_pos_raw': 'pct_pos_norm'}).copy()
raw_tmp['method'] = 'Raw'
frames.append(raw_tmp)

all_pop_shift = pd.concat(frames, ignore_index=True)
method_order_avail = [m for m in METHOD_ORDER_SHIFT if m in all_pop_shift['method'].unique()]

# Violin grid
all_markers_sorted = sorted(all_pop_shift['marker'].unique().tolist())
n_markers = len(all_markers_sorted)
ncols = 4
nrows = (n_markers + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4), sharey=False)
axes = axes.ravel()

for i, marker in enumerate(all_markers_sorted):
    ax = axes[i]
    sub = all_pop_shift[all_pop_shift['marker'] == marker]
    data_by_method = [sub[sub['method'] == m]['pct_pos_norm'].dropna().values
                      for m in method_order_avail]
    positions = np.arange(len(method_order_avail))

    parts = ax.violinplot(
        [d if len(d) > 0 else np.array([0.0]) for d in data_by_method],
        positions=positions,
        showmedians=True, showextrema=True, widths=0.65,
    )
    for body, m in zip(parts['bodies'], method_order_avail):
        body.set_facecolor(METHOD_COLORS_SHIFT[m])
        body.set_alpha(0.75)
        body.set_edgecolor('none')
    parts['cmedians'].set_color('black')
    parts['cmedians'].set_linewidth(2)
    for key in ('cmaxes', 'cmins'):
        parts[key].set_visible(False)
    parts['cbars'].set_linewidth(0.8)
    parts['cbars'].set_color('gray')

    for j, d in enumerate(data_by_method):
        if len(d) > 0:
            ax.scatter(j, np.nanmean(d), s=35, color='white',
                       edgecolor='black', linewidth=1.0, zorder=5)

    is_bim = marker in bimodal_names
    ax.set_title(marker, fontsize=9, fontweight='bold',
                 color='navy' if is_bim else 'black')
    ax.set_xticks(positions)
    ax.set_xticklabels(method_order_avail, rotation=45, ha='right', fontsize=6.5)
    ax.set_ylabel('% Positive', fontsize=7)
    ax.grid(True, alpha=0.3, axis='y')
    y_max = float(sub['pct_pos_norm'].max()) if len(sub) > 0 else 5.0
    ax.set_ylim(-2, max(y_max, 5.0) + 5)

for i in range(n_markers, len(axes)):
    axes[i].set_visible(False)

legend_patches = [Patch(facecolor=METHOD_COLORS_SHIFT[m], alpha=0.8, label=m)
                  for m in method_order_avail]
fig.legend(handles=legend_patches, loc='lower right', ncol=len(method_order_avail),
           fontsize=9, bbox_to_anchor=(0.99, 0.01), frameon=True)

plt.suptitle(
    'Positive Population (% positive cells) per Marker — All Methods\n'
    'Violin = sample distribution  ·  White dot = mean  ·  Black line = median\n'
    'Blue titles = bimodal markers',
    fontsize=11,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

# Mean % positive summary table
pivot_mean_shift = (
    all_pop_shift.groupby(['marker', 'method'])['pct_pos_norm']
    .mean()
    .unstack('method')[method_order_avail]
    .round(1)
)
print('\nMean % Positive Cells per Marker (across 20 samples):')
print('=' * 75)
print(pivot_mean_shift.to_string())

raw_ref = all_pop_shift[all_pop_shift['method'] == 'Raw']['pct_pos_norm'].mean()
print('\nOverall mean % positive (all markers × samples):')
for m in method_order_avail:
    overall = all_pop_shift[all_pop_shift['method'] == m]['pct_pos_norm'].mean()
    delta   = overall - raw_ref
    sign    = '+' if delta >= 0 else ''
    print(f'  {m:>14s}: {overall:.2f}%   (Δ from Raw: {sign}{delta:.2f}%)')

## 7. Histogram Comparison

Per-sample histogram PDFs for all markers. Saved to `histograms_shift/`.

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import colormaps

save_dir = 'histograms_shift'
os.makedirs(save_dir, exist_ok=True)

layers_to_plot = [
    ('normalized_affine', 'Affine only'),
    ('normalized_shift_a1.0', 'Shift (α=1.0)'),
]

sample_col = 'sample_id' if 'sample_id' in adata.obs.columns else 'batch'
sample_ids = adata.obs[sample_col].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

pdf_path = os.path.join(save_dir, 'shift_histograms.pdf')
n_cols = len(layers_to_plot)
rows_per_page = 4

with PdfPages(pdf_path) as pdf:
    page_markers = []
    for i, mname in enumerate(marker_names):
        page_markers.append(mname)
        if len(page_markers) == rows_per_page or i == len(marker_names) - 1:
            fig, axes = plt.subplots(len(page_markers), n_cols,
                                     figsize=(7 * n_cols, 4 * len(page_markers)))
            if len(page_markers) == 1:
                axes = axes.reshape(1, -1)
            if n_cols == 1:
                axes = axes.reshape(-1, 1)

            for row, mname in enumerate(page_markers):
                m_idx = marker_names.index(mname)
                for col, (layer_key, label) in enumerate(layers_to_plot):
                    ax = axes[row, col]
                    X_col = np.array(adata.layers[layer_key][:, m_idx]).ravel()
                    X_log = np.log1p(np.clip(X_col, 0, None))
                    for s in unique_samples:
                        mask = sample_ids == s
                        c, e = np.histogram(X_log[mask], bins=80)
                        ax.plot(e[:-1], c, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
                    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
                    ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
                    ax.set_xlabel('log1p intensity', fontsize=8)
                    if col == 0:
                        ax.set_ylabel('Count', fontsize=8)
                    if row == 0 and col == 0:
                        ax.legend(fontsize=5, ncol=3, frameon=False)
                    ax.spines[['top', 'right']].set_visible(False)

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)
            page_markers = []

print(f'PDF saved to: {pdf_path}')

## 8. kBET Evaluation

Compute kBET acceptance rate for 5 clinical groups pairing samples from different batches.

- **Higher kBET = better** (batches well-mixed in local neighborhoods)
- **Lower chi-square = better**

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

EVAL_LAYERS = ['normalized_affine', 'normalized_shift_a1.0']
if 'normalized_shift_a0.3' in adata.layers:
    EVAL_LAYERS.insert(1, 'normalized_shift_a0.3')

print(f'Evaluating layers: {EVAL_LAYERS}')

In [ ]:
all_kbet = {}

for layer_name in EVAL_LAYERS:
    print(f'\n{"-"*55}\n  {layer_name}\n{"-"*55}')
    adata_kbet = adata.copy()
    adata_kbet.X = adata.layers[layer_name].copy()

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}

    all_kbet[layer_name] = layer_res

print('\nDone.')

In [ ]:
print('=' * 80)
print('kBET Detailed Results — per clinical group')
print('=' * 80)

rows = []
for layer_name, res in all_kbet.items():
    for gname, vals in res.items():
        rows.append({
            'layer': layer_name,
            'group': gname,
            'kBET':  vals['kBET'],
            'chi2':  vals['chi2'],
            'p':     vals['p'],
        })

df_detail = pd.DataFrame(rows)
print(df_detail.to_string(index=False, float_format='%.4f'))

print('\n' + '=' * 80)
print('kBET Summary (mean across groups — higher kBET = better, lower chi2 = better, higher p = better)')
print('=' * 80)

summary_rows = []
for layer_name, res in all_kbet.items():
    kbets = [v['kBET'] for v in res.values() if not np.isnan(v['kBET'])]
    chi2s = [v['chi2'] for v in res.values() if not np.isnan(v['chi2'])]
    pvals = [v['p']    for v in res.values() if not np.isnan(v['p'])]
    if kbets:
        row = {
            'layer':     layer_name,
            'mean_kBET': float(np.mean(kbets)),
            'mean_chi2': float(np.mean(chi2s)),
            'mean_p':    float(np.mean(pvals)),
            'n_groups':  f'{len(kbets)}/{len(res)}',
        }
        summary_rows.append(row)
        print(f"  {layer_name:>30s}  kBET={row['mean_kBET']:.4f}  "
              f"chi2={row['mean_chi2']:.4f}  p={row['mean_p']:.4f}  ({row['n_groups']} groups)")

df_kbet = pd.DataFrame(summary_rows)

if len(df_kbet) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(df_kbet['layer'], df_kbet['mean_kBET'], color='steelblue', alpha=0.85)
    axes[0].set_ylabel('Mean kBET acceptance rate')
    axes[0].set_title('kBET (higher = better)')
    axes[0].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[0].grid(True, alpha=0.3, axis='y')

    axes[1].bar(df_kbet['layer'], df_kbet['mean_chi2'], color='salmon', alpha=0.85)
    axes[1].set_ylabel('Mean chi-square')
    axes[1].set_title('Chi-square (lower = better)')
    axes[1].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    axes[2].bar(df_kbet['layer'], df_kbet['mean_p'], color='mediumseagreen', alpha=0.85)
    axes[2].set_ylabel('Mean p-value')
    axes[2].set_title('p-value (higher = better)')
    axes[2].set_xticklabels(df_kbet['layer'], rotation=30, ha='right', fontsize=9)
    axes[2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()